In [0]:
dbutils.widgets.text(
    "input_path",
    "",
    "Input Path"
)

dbutils.widgets.text(
    "batch_id",
    "",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "source_load_type",
    "incremental",
    ["full", "incremental"],
    "Source Load Type"
)

dbutils.widgets.dropdown(
    "execution_mode",
    "standard",
    ["standard", "backfill"],
    "Execution Mode"
)

In [0]:


input_path = dbutils.widgets.get("input_path")
batch_id = dbutils.widgets.get("batch_id")
source_load_type = dbutils.widgets.get("source_load_type")
execution_mode = dbutils.widgets.get("execution_mode")

if not input_path:
    raise ValueError("input_path cannot be empty")

if not batch_id:
    raise ValueError("batch_id cannot be empty")

print("Input:", input_path)
print("Batch:", batch_id)
print("Source load type:", source_load_type)
print("Execution mode:", execution_mode)

In [0]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType
)

CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

LOG_TABLE = (
    f"`{CATALOG}`.`{SCHEMA}`."
    "`pipeline_execution_logs`"
)

QUARANTINE_TABLE = (
    f"`{CATALOG}`.`{SCHEMA}`."
    "`quarantine_records`"
)

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

In [0]:
owner_schema = StructType([
    StructField("login", StringType(), True)
])

repository_payload_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("full_name", StringType(), True),
    StructField("owner", owner_schema, True),
    StructField("description", StringType(), True),
    StructField("language", StringType(), True),
    StructField("stargazers_count", StringType(), True),
    StructField("forks_count", StringType(), True),
    StructField("open_issues_count", StringType(), True),
    StructField("default_branch", StringType(), True),
    StructField("archived", BooleanType(), True),
    StructField("disabled", BooleanType(), True),
    StructField("fork", BooleanType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("pushed_at", StringType(), True),
])

repository_schema = StructType([
    StructField(
        "source_repository",
        StringType(),
        False
    ),
    StructField(
        "fetched_at_utc",
        StringType(),
        False
    ),
    StructField(
        "payload",
        repository_payload_schema,
        True
    ),
    StructField(
        "_corrupt_record",
        StringType(),
        True
    ),
])

In [0]:
commit_person_schema = StructType([
    StructField("name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("date", StringType(), True),
])

git_commit_schema = StructType([
    StructField(
        "author",
        commit_person_schema,
        True
    ),
    StructField(
        "committer",
        commit_person_schema,
        True
    ),
    StructField(
        "message",
        StringType(),
        True
    ),
])

github_user_schema = StructType([
    StructField("login", StringType(), True)
])

commit_payload_schema = StructType([
    StructField("sha", StringType(), True),
    StructField(
        "commit",
        git_commit_schema,
        True
    ),
    StructField(
        "author",
        github_user_schema,
        True
    ),
    StructField(
        "committer",
        github_user_schema,
        True
    ),
])

commit_schema = StructType([
    StructField(
        "source_repository",
        StringType(),
        False
    ),
    StructField(
        "fetched_at_utc",
        StringType(),
        False
    ),
    StructField(
        "payload",
        commit_payload_schema,
        True
    ),
    StructField(
        "_corrupt_record",
        StringType(),
        True
    ),
])

## Issue Schema

In [0]:
issue_user_schema = StructType([
    StructField("login", StringType(), True)
])

pull_request_marker_schema = StructType([
    StructField("url", StringType(), True)
])

issue_payload_schema = StructType([
    StructField("id", StringType(), True),
    StructField("number", StringType(), True),
    StructField("state", StringType(), True),
    StructField("state_reason", StringType(), True),
    StructField("title", StringType(), True),
    StructField("user", issue_user_schema, True),
    StructField("comments", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("closed_at", StringType(), True),
    StructField(
        "pull_request",
        pull_request_marker_schema,
        True
    ),
])

issue_schema = StructType([
    StructField(
        "source_repository",
        StringType(),
        False
    ),
    StructField(
        "fetched_at_utc",
        StringType(),
        False
    ),
    StructField(
        "payload",
        issue_payload_schema,
        True
    ),
    StructField(
        "_corrupt_record",
        StringType(),
        True
    ),
])

## PR Schema

In [0]:
pr_user_schema = StructType([
    StructField("login", StringType(), True)
])

pr_payload_schema = StructType([
    StructField("id", StringType(), True),
    StructField("number", StringType(), True),
    StructField("state", StringType(), True),
    StructField("title", StringType(), True),
    StructField("user", pr_user_schema, True),
    StructField("draft", BooleanType(), True),
    StructField("created_at", StringType(), True),
    StructField("updated_at", StringType(), True),
    StructField("closed_at", StringType(), True),
    StructField("merged_at", StringType(), True),
    StructField(
        "merge_commit_sha",
        StringType(),
        True
    ),
])

pr_schema = StructType([
    StructField(
        "source_repository",
        StringType(),
        False
    ),
    StructField(
        "fetched_at_utc",
        StringType(),
        False
    ),
    StructField(
        "payload",
        pr_payload_schema,
        True
    ),
    StructField(
        "_corrupt_record",
        StringType(),
        True
    ),
])

## audit-log helper

In [0]:
def write_log(
    run_id,
    layer,
    dataset,
    batch_id,
    input_parameter,
    source_load_type,
    execution_mode,
    start_time,
    end_time,
    status,
    rows_read,
    rows_inserted,
    rows_updated,
    rows_quarantined,
    error_message=None
):
    log_data = [(
        run_id,
        "github_open_source_analytics",
        layer,
        dataset,
        batch_id,
        input_parameter,
        source_load_type,
        execution_mode,
        start_time,
        end_time,
        status,
        int(rows_read),
        int(rows_inserted),
        int(rows_updated),
        int(rows_quarantined),
        error_message
    )]

    log_schema = """
        run_id STRING,
        pipeline_name STRING,
        layer STRING,
        dataset STRING,
        batch_id STRING,
        input_parameter STRING,
        source_load_type STRING,
        execution_mode STRING,
        execution_start TIMESTAMP,
        execution_end TIMESTAMP,
        status STRING,
        rows_read LONG,
        rows_inserted LONG,
        rows_updated LONG,
        rows_quarantined LONG,
        error_message STRING
    """

    (
        spark.createDataFrame(
            log_data,
            schema=log_schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )

## Quarantine helper

In [0]:
def write_quarantine(
    invalid_df,
    dataset,
    batch_id,
    key_column,
    load_timestamp
):
    count = invalid_df.count()

    if count == 0:
        return 0

    columns_for_json = [
        F.col(c)
        for c in invalid_df.columns
        if c != "_corrupt_record"
    ]

    q_df = (
        invalid_df
        .withColumn(
            "error_reason",
            F.when(
                F.col("_corrupt_record").isNotNull(),
                F.lit("Malformed or non-conforming JSON")
            ).when(
                F.col(key_column).isNull(),
                F.lit("Required business key is NULL")
            ).otherwise(
                F.lit("Schema validation failure")
            )
        )
        .select(
            F.lit(dataset).alias("dataset"),
            F.lit("Raw-to-Bronze").alias("layer"),
            F.lit(batch_id).alias("batch_id"),
            F.col("source_file"),
            F.col(key_column)
                .cast("string")
                .alias("business_key"),
            F.col("error_reason"),
            F.coalesce(
                F.col("_corrupt_record"),
                F.to_json(
                    F.struct(*columns_for_json)
                )
            ).alias("record_json"),
            F.lit(load_timestamp)
                .cast("timestamp")
                .alias("load_timestamp")
        )
    )

    (
        q_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(QUARANTINE_TABLE)
    )

    return count

## Bronze Merge Helper

In [0]:
def merge_bronze(
    source_df,
    target_table,
    key_columns
):
    source_df = source_df.dropDuplicates(
        key_columns
    )

    existing_keys = (
        spark.table(target_table)
        .select(*key_columns)
    )

    rows_inserted = (
        source_df
        .select(*key_columns)
        .join(
            existing_keys,
            key_columns,
            "left_anti"
        )
        .count()
    )

    condition = " AND ".join(
        [
            f"t.`{key}` = s.`{key}`"
            for key in key_columns
        ]
    )

    target = DeltaTable.forName(
        spark,
        target_table
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            condition
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    return rows_inserted

## building repo RAW -> Bronze

In [0]:
def process_repositories():
    dataset = "repositories"

    run_id = str(uuid.uuid4())

    start_time = datetime.now(timezone.utc)

    load_timestamp = datetime.now(
        timezone.utc
    )

    rows_read = 0
    rows_inserted = 0
    rows_quarantined = 0

    file_path = (
        f"{input_path}/repositories.json.gz"
    )

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "bronze_repositories"
    )

    try:
        raw_df = (
            spark.read
            .schema(repository_schema)
            .option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .json(file_path)
        )

        rows_read = raw_df.count()

        flattened = raw_df.select(
            F.col("payload.id")
                .alias("repo_id"),

            F.col("source_repository"),

            F.col("payload.name")
                .alias("name"),

            F.col("payload.full_name")
                .alias("full_name"),

            F.col("payload.owner.login")
                .alias("owner_login"),

            F.col("payload.description")
                .alias("description"),

            F.col("payload.language")
                .alias("language"),

            F.col(
                "payload.stargazers_count"
            ).alias("stargazers_count"),

            F.col(
                "payload.forks_count"
            ).alias("forks_count"),
    
            F.col(
                "payload.open_issues_count"
            ).alias("open_issues_count"),

            F.col(
                "payload.default_branch"
            ).alias("default_branch"),

            F.col("payload.archived")
                .alias("archived"),

            F.col("payload.disabled")
                .alias("disabled"),

            F.col("payload.fork")
                .alias("fork"),

            F.col("payload.created_at")
                .alias("created_at_raw"),

            F.col("payload.updated_at")
                .alias("updated_at_raw"),

            F.col("payload.pushed_at")
                .alias("pushed_at_raw"),

            F.col("fetched_at_utc")
                .alias("fetched_at_utc_raw"),

            F.col("_corrupt_record"),

            F.col("_metadata.file_path")
                .alias("source_file")
        )

        invalid = flattened.filter(
            F.col("_corrupt_record").isNotNull()
            | F.col("repo_id").isNull()
        )

        rows_quarantined = write_quarantine(
            invalid,
            dataset,
            batch_id,
            "repo_id",
            load_timestamp
        )

        valid = (
            flattened
            .filter(
                F.col("_corrupt_record").isNull()
                & F.col("repo_id").isNotNull()
            )
            .drop("_corrupt_record")
            .withColumn(
                "batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "source_load_type",
                F.lit(source_load_type)
            )
            .withColumn(
                "execution_mode",
                F.lit(execution_mode)
            )
            .withColumn(
                "schema_version",
                F.lit("v1")
            )
            .withColumn(
                "load_timestamp",
                F.lit(load_timestamp)
                 .cast("timestamp")
            )
        )

        rows_inserted = merge_bronze(
            valid,
            target_table,
            ["repo_id", "batch_id"]
        )

        end_time = datetime.now(
            timezone.utc
        )

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "SUCCESS",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            None
        )

        print(
            f"{dataset}: SUCCESS | "
            f"read={rows_read}, "
            f"inserted={rows_inserted}, "
            f"quarantined={rows_quarantined}"
        )

    except Exception as error:
        end_time = datetime.now(
            timezone.utc
        )

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "FAILURE",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            str(error)
        )

        raise

### Commits Table func

In [0]:
def process_commits():
    dataset = "commits"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)
    load_timestamp = datetime.now(timezone.utc)

    rows_read = 0
    rows_inserted = 0
    rows_quarantined = 0

    file_path = f"{input_path}/commits.json.gz"

    target_table = (
        f"{CATALOG}.{SCHEMA}.bronze_commits"
    )

    try:
        raw_df = (
            spark.read
            .schema(commit_schema)
            .option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .json(file_path)
        )

        rows_read = raw_df.count()

        flattened = raw_df.select(
            F.col("payload.sha")
                .alias("commit_sha"),

            F.col("source_repository"),

            F.col("payload.author.login")
                .alias("author_login"),

            F.col("payload.commit.author.name")
                .alias("author_name"),

            F.col("payload.commit.author.email")
                .alias("author_email"),

            F.col("payload.commit.author.date")
                .alias("author_date_raw"),

            F.col("payload.committer.login")
                .alias("committer_login"),

            F.col(
                "payload.commit.committer.name"
            ).alias("committer_name"),

            F.col(
                "payload.commit.committer.email"
            ).alias("committer_email"),

            F.col(
                "payload.commit.committer.date"
            ).alias("committer_date_raw"),

            F.col("payload.commit.message")
                .alias("message"),

            F.col("fetched_at_utc")
                .alias("fetched_at_utc_raw"),

            F.col("_corrupt_record"),

            F.col("_metadata.file_path")
                .alias("source_file")
        )

        invalid = flattened.filter(
            F.col("_corrupt_record").isNotNull()
            | F.col("commit_sha").isNull()
        )

        rows_quarantined = write_quarantine(
            invalid,
            dataset,
            batch_id,
            "commit_sha",
            load_timestamp
        )

        valid = (
            flattened
            .filter(
                F.col("_corrupt_record").isNull()
                & F.col("commit_sha").isNotNull()
            )
            .drop("_corrupt_record")
            .withColumn(
                "batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "source_load_type",
                F.lit(source_load_type)
            )
            .withColumn(
                "execution_mode",
                F.lit(execution_mode)
            )
            .withColumn(
                "schema_version",
                F.lit("v1")
            )
            .withColumn(
                "load_timestamp",
                F.lit(load_timestamp)
                 .cast("timestamp")
            )
        )

        rows_inserted = merge_bronze(
            valid,
            target_table,
            ["commit_sha", "batch_id"]
        )

        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "SUCCESS",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            None
        )

        print(
            f"{dataset}: SUCCESS | "
            f"read={rows_read}, "
            f"inserted={rows_inserted}, "
            f"quarantined={rows_quarantined}"
        )

    except Exception as error:
        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "FAILURE",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            str(error)
        )

        raise

### Issue Table func

In [0]:
def process_issues():
    dataset = "issues"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)
    load_timestamp = datetime.now(timezone.utc)

    rows_read = 0
    rows_inserted = 0
    rows_quarantined = 0

    file_path = f"{input_path}/issues.json.gz"

    target_table = (
        f"{CATALOG}.{SCHEMA}.bronze_issues"
    )

    try:
        raw_df = (
            spark.read
            .schema(issue_schema)
            .option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .json(file_path)
        )

        rows_read = raw_df.count()

        flattened = raw_df.select(
            F.col("payload.id")
                .alias("issue_id"),

            F.col("payload.number")
                .alias("issue_number"),

            F.col("source_repository"),

            F.col("payload.state")
                .alias("state"),

            F.col("payload.state_reason")
                .alias("state_reason"),

            F.col("payload.title")
                .alias("title"),

            F.col("payload.user.login")
                .alias("user_login"),

            F.col("payload.comments")
                .alias("comments_raw"),

            F.col("payload.created_at")
                .alias("created_at_raw"),

            F.col("payload.updated_at")
                .alias("updated_at_raw"),

            F.col("payload.closed_at")
                .alias("closed_at_raw"),

            F.col("payload.pull_request")
                .isNotNull()
                .alias("is_pull_request"),

            F.col("fetched_at_utc")
                .alias("fetched_at_utc_raw"),

            F.col("_corrupt_record"),

            F.col("_metadata.file_path")
                .alias("source_file")
        )

        invalid = flattened.filter(
            F.col("_corrupt_record").isNotNull()
            | F.col("issue_id").isNull()
        )

        rows_quarantined = write_quarantine(
            invalid,
            dataset,
            batch_id,
            "issue_id",
            load_timestamp
        )

        valid = (
            flattened
            .filter(
                F.col("_corrupt_record").isNull()
                & F.col("issue_id").isNotNull()
            )
            .drop("_corrupt_record")
            .withColumn(
                "batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "source_load_type",
                F.lit(source_load_type)
            )
            .withColumn(
                "execution_mode",
                F.lit(execution_mode)
            )
            .withColumn(
                "schema_version",
                F.lit("v1")
            )
            .withColumn(
                "load_timestamp",
                F.lit(load_timestamp)
                 .cast("timestamp")
            )
        )

        rows_inserted = merge_bronze(
            valid,
            target_table,
            ["issue_id", "batch_id"]
        )

        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "SUCCESS",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            None
        )

        print(
            f"{dataset}: SUCCESS | "
            f"read={rows_read}, "
            f"inserted={rows_inserted}, "
            f"quarantined={rows_quarantined}"
        )

    except Exception as error:
        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "FAILURE",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            str(error)
        )

        raise

### PR table func

In [0]:
def process_pull_requests():
    dataset = "pull_requests"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(timezone.utc)
    load_timestamp = datetime.now(timezone.utc)

    rows_read = 0
    rows_inserted = 0
    rows_quarantined = 0

    file_path = (
        f"{input_path}/pull_requests.json.gz"
    )

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "bronze_pull_requests"
    )

    try:
        raw_df = (
            spark.read
            .schema(pr_schema)
            .option("multiLine", "true")
            .option("mode", "PERMISSIVE")
            .option(
                "columnNameOfCorruptRecord",
                "_corrupt_record"
            )
            .json(file_path)
        )

        rows_read = raw_df.count()

        flattened = raw_df.select(
            F.col("payload.id")
                .alias("pr_id"),

            F.col("payload.number")
                .alias("pr_number"),

            F.col("source_repository"),

            F.col("payload.state")
                .alias("state"),

            F.col("payload.title")
                .alias("title"),

            F.col("payload.user.login")
                .alias("user_login"),

            F.col("payload.draft")
                .alias("draft"),

            F.col("payload.created_at")
                .alias("created_at_raw"),

            F.col("payload.updated_at")
                .alias("updated_at_raw"),

            F.col("payload.closed_at")
                .alias("closed_at_raw"),

            F.col("payload.merged_at")
                .alias("merged_at_raw"),

            F.col("payload.merge_commit_sha")
                .alias("merge_commit_sha"),

            F.col("fetched_at_utc")
                .alias("fetched_at_utc_raw"),

            F.col("_corrupt_record"),

            F.col("_metadata.file_path")
                .alias("source_file")
        )

        invalid = flattened.filter(
            F.col("_corrupt_record").isNotNull()
            | F.col("pr_id").isNull()
        )

        rows_quarantined = write_quarantine(
            invalid,
            dataset,
            batch_id,
            "pr_id",
            load_timestamp
        )

        valid = (
            flattened
            .filter(
                F.col("_corrupt_record").isNull()
                & F.col("pr_id").isNotNull()
            )
            .drop("_corrupt_record")
            .withColumn(
                "batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "source_load_type",
                F.lit(source_load_type)
            )
            .withColumn(
                "execution_mode",
                F.lit(execution_mode)
            )
            .withColumn(
                "schema_version",
                F.lit("v1")
            )
            .withColumn(
                "load_timestamp",
                F.lit(load_timestamp)
                 .cast("timestamp")
            )
        )

        rows_inserted = merge_bronze(
            valid,
            target_table,
            ["pr_id", "batch_id"]
        )

        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "SUCCESS",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            None
        )

        print(
            f"{dataset}: SUCCESS | "
            f"read={rows_read}, "
            f"inserted={rows_inserted}, "
            f"quarantined={rows_quarantined}"
        )

    except Exception as error:
        end_time = datetime.now(timezone.utc)

        write_log(
            run_id,
            "Raw-to-Bronze",
            dataset,
            batch_id,
            file_path,
            source_load_type,
            execution_mode,
            start_time,
            end_time,
            "FAILURE",
            rows_read,
            rows_inserted,
            0,
            rows_quarantined,
            str(error)
        )

        raise

In [0]:
print("Starting Raw -> Bronze pipeline")

process_repositories()
process_commits()
process_issues()
process_pull_requests()

print("Raw -> Bronze pipeline finished")

In [0]:
tables = [
    "bronze_repositories",
    "bronze_commits",
    "bronze_issues",
    "bronze_pull_requests",
]

for table in tables:
    count = spark.table(
        f"{CATALOG}.{SCHEMA}.{table}"
    ).count()

    print(table, count)